# Day 18: Missing Values in Pandas

**Dataset:** `House_Prices_Dataset.csv`

This notebook identifies missing values in a dataset and practices handling them using
techniques such as dropping and filling missing values.

**Note:** the real `House_Prices_Dataset.csv` file has **no missing values at all** (confirmed
in Section 2 below). Since this task is specifically about missing-value handling, Section 3
onward works with a copy of the real data that has a small number of values deliberately set
to missing (`NaN`), clearly labeled throughout, so the techniques can be genuinely
demonstrated rather than shown on data that has nothing to clean.

## 1. Import Libraries and Load the Dataset

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("../datasets/House_Prices_Dataset.csv")

# Preview the data
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


## 2. Missing-Value Summary (Real Data)

In [2]:
print(df.shape)
df.isna().sum()

(545, 13)


price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

As the summary above shows, every column has **0** missing values in the real dataset, implying that
this is a genuinely clean, complete dataset.

## 3. Creating a Working Copy with Simulated Missing Values

To practice the required techniques, a fixed random seed is used to set a small number of
real values to `NaN` in a copy of the DataFrame (`df` itself is left untouched).

In [3]:
np.random.seed(42)

df_missing = df.copy()

# Randomly blank out ~5% of values in a few real columns
for column in ["price", "area", "bedrooms", "furnishingstatus"]:
    missing_indices = df_missing.sample(frac=0.05, random_state=42).index
    df_missing.loc[missing_indices, column] = np.nan

df_missing.head(10)

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000.0,7420.0,4.0,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000.0,8960.0,4.0,4,4,yes,no,no,no,yes,3,no,furnished
2,NaN,NaN,NaN,2,2,yes,no,yes,no,no,2,yes,NaN
3,12215000.0,7500.0,4.0,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000.0,7420.0,4.0,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000.0,7500.0,3.0,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,NaN,NaN,NaN,3,4,yes,no,no,no,yes,2,yes,NaN
7,10150000.0,16200.0,5.0,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000.0,8100.0,4.0,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000.0,5750.0,3.0,2,4,yes,yes,no,no,yes,1,yes,unfurnished


## 4. Missing-Value Summary (Simulated Data)

In [4]:
missing_counts = df_missing.isna().sum()
missing_percent = (df_missing.isna().sum() / len(df_missing) * 100).round(2)

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})

missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_percent
price,27,4.95
area,27,4.95
bedrooms,27,4.95
furnishingstatus,27,4.95


In [5]:
print("Total missing cells:", df_missing.isna().sum().sum())
print("Total rows with at least one missing value:", df_missing.isna().any(axis=1).sum())

Total missing cells: 108
Total rows with at least one missing value: 27


## 5. Examples Using `isna()` and `notna()`

**`isna()` on a single column — returns True/False for each value**

In [6]:
print(df_missing["price"].isna().head(10))

0    False
1    False
2     True
3    False
4    False
5    False
6     True
7    False
8    False
9    False
Name: price, dtype: bool


**Using `isna()` to filter down to only the rows with a missing value in a column**

In [7]:
missing_price_rows = df_missing[df_missing["price"].isna()]
print(len(missing_price_rows), "rows with a missing price")
missing_price_rows.head()

27 rows with a missing price


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
2,NaN,NaN,NaN,2,2,yes,no,yes,no,no,2,yes,NaN
6,NaN,NaN,NaN,3,4,yes,no,no,no,yes,2,yes,NaN
55,NaN,NaN,NaN,1,2,yes,no,no,no,yes,1,no,NaN
72,NaN,NaN,NaN,1,4,yes,no,no,no,yes,0,yes,NaN
77,NaN,NaN,NaN,2,3,yes,no,no,no,yes,0,yes,NaN


**`notna()` — the opposite of `isna()`, True where a value IS present**

In [8]:
print(df_missing["area"].notna().head(10))

0     True
1     True
2    False
3     True
4     True
5     True
6    False
7     True
8     True
9     True
Name: area, dtype: bool


**Using `notna()` to filter down to only the rows that DO have a value**

In [9]:
valid_area_rows = df_missing[df_missing["area"].notna()]
print(len(valid_area_rows), "rows with a non-missing area (out of", len(df_missing), "total)")

518 rows with a non-missing area (out of 545 total)


**`isna()` on the whole DataFrame at once, then summed per row**

In [10]:
rows_with_any_missing = df_missing[df_missing.isna().any(axis=1)]
print(len(rows_with_any_missing), "rows have at least one missing value")
rows_with_any_missing.head()

27 rows have at least one missing value


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
2,NaN,NaN,NaN,2,2,yes,no,yes,no,no,2,yes,NaN
6,NaN,NaN,NaN,3,4,yes,no,no,no,yes,2,yes,NaN
55,NaN,NaN,NaN,1,2,yes,no,no,no,yes,1,no,NaN
72,NaN,NaN,NaN,1,4,yes,no,no,no,yes,0,yes,NaN
77,NaN,NaN,NaN,2,3,yes,no,no,no,yes,0,yes,NaN


## 6. Comparing Dropping and Filling Strategies

**Strategy 1 — `dropna()`: drop every row that has ANY missing value**

In [11]:
dropped_any = df_missing.dropna()
print("Original rows:", len(df_missing))
print("Rows remaining after dropna():", len(dropped_any))
print("Rows removed:", len(df_missing) - len(dropped_any))

Original rows: 545
Rows remaining after dropna(): 518
Rows removed: 27


**Strategy 2 — `dropna(subset=...)`: drop rows only if a SPECIFIC column is missing**

In [12]:
dropped_price_only = df_missing.dropna(subset=["price"])
print("Rows remaining after dropping only missing 'price':", len(dropped_price_only))

Rows remaining after dropping only missing 'price': 518


**Strategy 3 — `fillna()` with a constant placeholder**

In [13]:
filled_constant = df_missing.copy()
filled_constant["furnishingstatus"] = filled_constant["furnishingstatus"].fillna("unknown")
print(filled_constant["furnishingstatus"].isna().sum(), "missing values remaining in furnishingstatus")

0 missing values remaining in furnishingstatus


**Strategy 4 — `fillna()` with the median for numerical columns**

The median is generally safer than the mean for filling numeric columns, since it isn't
pulled up or down by outliers (very expensive or very cheap houses, for example).

In [14]:
filled_median = df_missing.copy()

price_median = filled_median["price"].median()
area_median = filled_median["area"].median()

filled_median["price"] = filled_median["price"].fillna(price_median)
filled_median["area"] = filled_median["area"].fillna(area_median)

print("Price median used:", price_median)
print("Area median used:", area_median)
print("Remaining missing values in price:", filled_median["price"].isna().sum())
print("Remaining missing values in area:", filled_median["area"].isna().sum())

Price median used: 4357500.0
Area median used: 4580.0
Remaining missing values in price: 0
Remaining missing values in area: 0


**Comparing mean vs median for `price`, to see why the choice matters**

In [15]:
print("Mean price:", round(df_missing["price"].mean(), 2))
print("Median price:", df_missing["price"].median())
print("The mean is higher than the median, meaning a few high-priced houses are pulling the")
print("average up - filling with the median avoids skewing the filled values toward those outliers.")

Mean price: 4777255.68
Median price: 4357500.0
The mean is higher than the median, meaning a few high-priced houses are pulling the
average up - filling with the median avoids skewing the filled values toward those outliers.


**Strategy 5 — `fillna()` with the mode (most frequent value) for a categorical column**

In [16]:
filled_mode = df_missing.copy()
bedrooms_mode = filled_mode["bedrooms"].mode()[0]
filled_mode["bedrooms"] = filled_mode["bedrooms"].fillna(bedrooms_mode)

print("Most common number of bedrooms:", bedrooms_mode)
print("Remaining missing values in bedrooms:", filled_mode["bedrooms"].isna().sum())

Most common number of bedrooms: 3.0
Remaining missing values in bedrooms: 0


**Comparing the two approaches on the same column (`price`)**

| Approach | Rows kept | What happens to the missing values |
|---|---|---|
| `dropna()` (all columns) | fewer rows | Every row with ANY missing value is removed entirely |
| `dropna(subset=["price"])` | more rows than above | Only rows missing `price` specifically are removed |
| `fillna(median)` | all rows kept | Missing values are replaced with a calculated estimate |
| `fillna(mode)` / `fillna("unknown")` | all rows kept | Missing values are replaced with the most common value or a placeholder label |


## 7. A Cleaned Version of the Dataset

Combining the strategies above: fill the numeric columns (`price`, `area`, `bedrooms`) with
their median, and fill the categorical column (`furnishingstatus`) with its mode, so no rows
need to be dropped at all.

In [17]:
df_cleaned = df_missing.copy()

for column in ["price", "area", "bedrooms"]:
    df_cleaned[column] = df_cleaned[column].fillna(df_cleaned[column].median())

df_cleaned["furnishingstatus"] = df_cleaned["furnishingstatus"].fillna(
    df_cleaned["furnishingstatus"].mode()[0]
)

print("Missing values remaining in the cleaned dataset:")
print(df_cleaned.isna().sum().sum())
df_cleaned.head(10)

Missing values remaining in the cleaned dataset:
0


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000.0,7420.0,4.0,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000.0,8960.0,4.0,4,4,yes,no,no,no,yes,3,no,furnished
2,4357500.0,4580.0,3.0,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000.0,7500.0,4.0,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000.0,7420.0,4.0,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000.0,7500.0,3.0,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,4357500.0,4580.0,3.0,3,4,yes,no,no,no,yes,2,yes,semi-furnished
7,10150000.0,16200.0,5.0,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000.0,8100.0,4.0,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000.0,5750.0,3.0,2,4,yes,yes,no,no,yes,1,yes,unfurnished


## Outcome

This notebook practiced identifying and handling missing values using the
`House_Prices_Dataset.csv` file. The real dataset turned out to have zero missing values, so
after confirming that with `isna().sum()`, a working copy with a small number of deliberately
simulated missing values was used to genuinely practice the task's techniques. I built a
missing-value summary table showing both the count and percentage missing per column, and used
`isna()`/`notna()` both on individual columns and across the whole DataFrame to filter rows
with and without missing data. Comparing dropping (`dropna()`, with and without a `subset`)
against filling (`fillna()` with a constant, the median, and the mode) showed that dropping
loses entire rows of otherwise-valid data, while filling keeps every row but requires choosing
a sensible replacement value. Also, the median is generally safer than the mean for a
skewed numeric column like `price`, since it isn't pulled toward outliers. Combining median and
mode filling produced a fully cleaned dataset with no missing values and no rows lost.